# 🔬 Supervised Fine-Tuning FLAN-T5 on Clinical Consultations

This notebook demonstrates fine-tuning Google's **FLAN-T5** sequence-to-sequence model to extract structured Electronic Health Record (EHR) categories from doctor-patient conversation transcripts.

### Workflow Overview:
1. **Environment Setup & Data Loading**: Loading pre-split clinical consultation dialogues.
2. **Prompt Formatting & Tokenization**: Structuring multi-turn conversations and formatting targets.
3. **Model Configuration**: Initializing `google/flan-t5-base` with dynamic sequence length.
4. **Training with Seq2SeqTrainer**: Parameter configuration and gradient accumulation.
5. **Evaluation & Generation**: Generating structured summaries and inspecting outputs.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Add repository root to python path
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.dataset import load_clinical_splits, format_prompt
from src.training.collator import ClinicalSeq2SeqCollator
from src.evaluation.metrics import compute_clinical_metrics

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

## 1. Inspect Clinical Dataset Splits

In [ ]:
train_path = "../data/processed/train.csv"
val_path = "../data/processed/val.csv"
test_path = "../data/processed/test.csv"

train_df = pd.read_csv(train_path)
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(pd.read_csv(val_path))}")
print(f"Test samples: {len(pd.read_csv(test_path))}")

print("\n--- Sample Training Pair ---")
print("Input Prompt:", train_df.iloc[0]['input'][:200], "...")
print("Target Answer:", train_df.iloc[0]['target'])

## 2. Load Model and Tokenizer

In [ ]:
model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
print(f"Successfully loaded {model_id} with {model.num_parameters():,} parameters.")

## 3. Tokenize Dataset with Dynamic Padding

In [ ]:
splits = load_clinical_splits(
    train_path=train_path,
    val_path=val_path,
    test_path=test_path,
    tokenizer=tokenizer,
    max_input_length=512,
    max_target_length=128,
    pad_to_max=False
)
print("Tokenized features:", splits['train'][0].keys())

## 4. Configure Training Arguments & Trainer

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/flan-t5-base-notebook",
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

data_collator = ClinicalSeq2SeqCollator(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=splits['train'],
    eval_dataset=splits['validation'],
    data_collator=data_collator,
    processing_class=tokenizer
)

# To start fine-tuning, execute:
# trainer.train()
# trainer.save_model("../models/flan-t5-base-finetuned")

## 5. Inference Verification on Clinical Case

In [ ]:
sample_context = """What is your name? My name is Selena Smith. What brings you in today?

I've been experiencing lower back pain for about three weeks. It started as a dull ache but has recently

become sharp, especially when I bend or sit for long periods. Do you have any other symptoms? Yes, occasionally

I feel tingling down my right leg. Past medical history: migraines managed with ibuprofen. Surgeries: cyst removal 3 years ago."""

questions = [
    "List the patient's chief complaints with comma separated value.",
    "Summarize the patient's past medical history.",
    "What is the patient's hospitalization and surgical history?"
]

model.eval()
for q in questions:
    prompt = format_prompt(question=q, context=sample_context)
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model.generate(inputs['input_ids'], max_new_tokens=50, num_beams=2)
    ans = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Q: {q}\nA: {ans}\n")